# Ablation RO-A — initial condition only

Solid-body rotation about the domain centre. The exact solution is a circle
rotating rigidly, available in closed form, and it remains a signed distance
function for all time -- so the eikonal constraint is unambiguous here.

Velocity fixed; only the initial interface varies. This is the standard
single-input-function protocol in the operator literature, so the numbers are
comparable to it.

Three arms x three seeds, 16 training instances.

| | |
|---|---|
| grid | 64 x 64 x 32 |
| test set | 100 held-out instances |
| Adam | 20,000 steps, batch 4 |
| model | width 20, modes (12,12,8) |
| eikonal weight | SAW, zero-seeded |
| seeds | 42, 43, 44 |

No L-BFGS stage: Adam at 20k reaches the same place on these benchmarks, and
adding a refinement that helps one arm more than another would confound the
comparison.

The eikonal weight uses `--saw_init zero`. Under a hard initial condition the
field starts exactly at phi_0, which IS a signed distance function, so g_eik
starts near zero, the raw ratio saturates, and the published seeding pins the
weight at its clamp for thousands of steps. Seeding at zero lets it climb as
the field genuinely departs from phi_0.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/NORO
!ls *.py

/content/drive/MyDrive/NORO
collect_results.py  fno3d.py	  ro_family.py	     verify_residual.py
eval_checkpoint.py  residuals.py  train_operator.py  visualize.py


In [3]:
import torch; print(torch.cuda.get_device_name(0))

Tesla T4


### Sanity check

The exact solution should drive the transport residual toward zero under mesh
refinement while a field violating transport stays flat, with the ratio growing
as the mesh refines.

In [ ]:
!python verify_residual.py

         h          exact         frozen      ratio
    0.0625      5.032e-03      1.483e-01       29.5
    0.0312      1.529e-03      1.510e-01       98.7  (x3.3)
    0.0156      4.471e-04      1.519e-01      339.7  (x3.4)
    0.0078      1.267e-04      1.522e-01     1201.8  (x3.5)


---
## Training

### data-free (physics only)

In [ ]:
!python train_operator.py --loss strong --saw --saw_init zero --steps 20000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 0/16 instances (data-free)  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 3.184e-03 | train  39.276% | test  39.246% | mass  50.35% | drift  51.91% | w_eik 7.142e-02 | 0.6m
   500 loss 2.997e-03 | train  39.278% | test  39.255% | mass  56.67% | drift  58.44% | w_eik 2.391e-01 | 1.1m
   750 loss 1.987e-03 | train  39.816% | test  39.795% | mass  63.79% | drift  65.80% | w_eik 3.703e-01 | 1.7m
  1000 loss 1.777e-03 | train  40.007% | test  39.988% | mass  65.92% | drift  68.00% | w_eik 4.263e-01 | 2.2m
  1250 loss 1.538e-03 | train  39.930% | test  39.916% | mass  64.83% | drift  66.88% | w_eik 3.883e-01 | 2.8m
  1500 loss 1.376e-03 | train  39.101% | test  39.077% | mass  59.52% | drift  61.39% | w_eik 3.348e-01 | 3.4m
  1750 loss 1.160e-03 | train  38.771% | test  38.757% | mass  59.41% | drift  61.27% | w_eik 2.820e-01 | 3.9m
  2000 loss

In [ ]:
!python train_operator.py --loss strong --saw --saw_init zero --steps 20000 --n_train 16 --n_test 100 --seed 43

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 0/16 instances (data-free)  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 3.069e-03 | train  39.492% | test  39.596% | mass  59.48% | drift  61.45% | w_eik 6.359e-02 | 0.6m
   500 loss 2.566e-03 | train  39.304% | test  39.409% | mass  58.01% | drift  59.94% | w_eik 2.163e-01 | 1.1m
   750 loss 1.864e-03 | train  39.845% | test  39.965% | mass  66.24% | drift  68.41% | w_eik 3.410e-01 | 1.7m
  1000 loss 1.527e-03 | train  40.145% | test  40.263% | mass  66.57% | drift  68.75% | w_eik 3.859e-01 | 2.2m
  1250 loss 1.628e-03 | train  40.348% | test  40.474% | mass  69.49% | drift  71.77% | w_eik 3.926e-01 | 2.8m
  1500 loss 1.321e-03 | train  40.019% | test  40.142% | mass  65.93% | drift  68.10% | w_eik 3.630e-01 | 3.4m
  1750 loss 1.223e-03 | train  39.634% | test  39.756% | mass  61.23% | drift  63.25% | w_eik 3.176e-01 | 3.9m
  2000 loss

In [ ]:
!python train_operator.py --loss strong --saw --saw_init zero --steps 20000 --n_train 16 --n_test 100 --seed 44

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 0/16 instances (data-free)  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 3.207e-03 | train  39.313% | test  39.290% | mass  50.50% | drift  52.10% | w_eik 5.821e-02 | 0.6m
   500 loss 2.451e-03 | train  39.541% | test  39.527% | mass  61.82% | drift  63.80% | w_eik 1.427e-01 | 1.1m
   750 loss 1.660e-03 | train  39.732% | test  39.725% | mass  64.32% | drift  66.37% | w_eik 1.461e-01 | 1.6m
  1000 loss 1.462e-03 | train  39.985% | test  39.984% | mass  66.18% | drift  68.29% | w_eik 1.436e-01 | 2.2m
  1250 loss 1.304e-03 | train  39.024% | test  39.028% | mass  62.62% | drift  64.62% | w_eik 1.300e-01 | 2.7m
  1500 loss 9.166e-04 | train  37.686% | test  37.700% | mass  59.40% | drift  61.29% | w_eik 1.115e-01 | 3.3m
  1750 loss 7.674e-04 | train  36.292% | test  36.307% | mass  59.30% | drift  61.19% | w_eik 9.187e-02 | 3.8m
  2000 loss

In [ ]:
!python collect_results.py

arm         v        n       grid  steps sd    train     test     mass    min
-----------------------------------------------------------------------------
data-free   fixed   16    64^2x32  20000 42   4.930%   5.258%    5.50%   45.0
data-free   fixed   16    64^2x32  20000 43   2.956%   3.258%    4.78%   45.0
data-free   fixed   16    64^2x32  20000 44   5.254%   5.632%    4.03%   43.7

arm         v        n  steps seeds  test mean     std  mass mean     std
-------------------------------------------------------------------------
data-free   fixed   16  20000     3     4.716%  1.277%      4.77%   0.74%


### supervised (labels only)

In [ ]:
!python train_operator.py --loss supervised --steps 20000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=supervised
benchmark: RO  (velocity fixed)
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 8.356e-03 | train  26.782% | test  27.193% | mass  67.28% | drift  69.43% | 0.5m
   500 loss 2.188e-03 | train  12.769% | test  13.760% | mass  28.77% | drift  29.68% | 1.0m
   750 loss 4.970e-04 | train   6.242% | test   7.524% | mass  17.02% | drift  17.57% | 1.6m
  1000 loss 3.117e-04 | train   4.520% | test   5.986% | mass  14.29% | drift  14.77% | 2.1m
  1250 loss 1.762e-04 | train   3.688% | test   5.200% | mass  12.61% | drift  13.00% | 2.6m
  1500 loss 1.249e-04 | train   3.065% | test   4.596% | mass  12.28% | drift  12.71% | 3.1m
  1750 loss 1.190e-04 | train   2.788% | test   4.367% | mass  11.56% | drift  11.92% | 3.6m
  2000 loss 5.800e-05 | train   2.194% | test   3.876% | mass  11.12% | drift  11.50% | 4.2m
  2250 loss 5.168e-05 | train   1.986% | test   3.709% | mass  10.47% | drift  10.77% | 4.7m
  2500 loss 6.170e-05 | train   

In [4]:
!python train_operator.py --loss supervised --steps 20000 --n_train 16 --n_test 100 --seed 43

device=cuda  grid=64x64x32  loss=supervised
benchmark: RO  (velocity fixed)
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 8.777e-03 | train  26.847% | test  27.212% | mass  70.15% | drift  72.43% | 0.5m
   500 loss 2.520e-03 | train  13.476% | test  15.185% | mass  39.55% | drift  40.82% | 1.0m
   750 loss 5.145e-04 | train   6.305% | test   8.148% | mass  22.49% | drift  23.22% | 1.6m
  1000 loss 3.035e-04 | train   4.773% | test   6.772% | mass  18.71% | drift  19.38% | 2.1m
  1250 loss 2.370e-04 | train   4.023% | test   6.028% | mass  16.24% | drift  16.82% | 2.6m
  1500 loss 1.633e-04 | train   3.422% | test   5.420% | mass  14.89% | drift  15.42% | 3.1m
  1750 loss 9.851e-05 | train   2.888% | test   4.948% | mass  13.64% | drift  14.14% | 3.6m
  2000 loss 8.097e-05 | train   2.472% | test   4.533% | mass  13.11% | drift  13.60% | 4.2m
  2250 loss 5.465e-05 | train   2.168% | test   4.194% | mass  12.70% | drift  13.14% | 4.7m
  2500 loss 4.357e-05 | train   

In [5]:
!python train_operator.py --loss supervised --steps 20000 --n_train 16 --n_test 100 --seed 44

device=cuda  grid=64x64x32  loss=supervised
benchmark: RO  (velocity fixed)
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 1.003e-02 | train  26.716% | test  26.827% | mass  71.55% | drift  73.86% | 0.5m
   500 loss 1.560e-03 | train  10.443% | test  11.443% | mass  23.53% | drift  24.29% | 1.0m
   750 loss 3.540e-04 | train   5.687% | test   7.100% | mass  14.78% | drift  15.24% | 1.6m
  1000 loss 2.839e-04 | train   4.368% | test   6.050% | mass  14.21% | drift  14.66% | 2.1m
  1250 loss 1.773e-04 | train   3.488% | test   5.259% | mass  14.07% | drift  14.55% | 2.6m
  1500 loss 1.133e-04 | train   2.839% | test   4.756% | mass  14.07% | drift  14.54% | 3.1m
  1750 loss 7.628e-05 | train   2.340% | test   4.378% | mass  13.19% | drift  13.61% | 3.7m
  2000 loss 7.008e-05 | train   2.139% | test   4.109% | mass  13.32% | drift  13.72% | 4.2m
  2250 loss 3.721e-05 | train   1.824% | test   3.940% | mass  12.70% | drift  13.09% | 4.7m
  2500 loss 3.608e-05 | train   

In [6]:
!python collect_results.py

arm         v        n       grid  steps sd    train     test     mass    min
-----------------------------------------------------------------------------
hybrid-8    fixed   16    64^2x32  20000 42   1.184%   1.845%    4.10%   44.5
data-free   fixed   16    64^2x32  20000 42   4.930%   5.258%    5.50%   45.0
data-free   fixed   16    64^2x32  20000 43   2.956%   3.258%    4.78%   45.0
data-free   fixed   16    64^2x32  20000 44   5.254%   5.632%    4.03%   43.7
supervised  fixed   16    64^2x32  20000 42   0.402%   2.441%    7.97%   41.7
supervised  fixed   16    64^2x32  20000 43   0.429%   2.611%    8.87%   41.7
supervised  fixed   16    64^2x32  20000 44   0.361%   2.811%   10.54%   41.8

arm         v        n  steps seeds  test mean     std  mass mean     std
-------------------------------------------------------------------------
data-free   fixed   16  20000     3     4.716%  1.277%      4.77%   0.74%
supervised  fixed   16  20000     3     2.621%  0.185%      9.13%   1.30%


### hybrid (8 of 16 labelled)

In [ ]:
!python train_operator.py --loss strong --saw --saw_init zero --n_labelled 8 --steps 20000 --n_train 16 --n_test 100 --seed 42

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 8/16 instances  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 1.953e-02 | train  33.919% | test  33.995% | mass  51.25% | drift  52.83% | data 1.46e-02 | w_eik 1.462e-01 | 0.6m
   500 loss 1.760e-02 | train  33.152% | test  33.266% | mass  46.38% | drift  47.80% | data 1.23e-02 | w_eik 3.350e-01 | 1.1m
   750 loss 1.485e-02 | train  30.288% | test  30.613% | mass  50.77% | drift  52.34% | data 8.39e-03 | w_eik 4.792e-01 | 1.7m
  1000 loss 1.201e-02 | train  24.449% | test  24.996% | mass  38.64% | drift  39.81% | data 5.71e-03 | w_eik 4.837e-01 | 2.2m
  1250 loss 7.590e-03 | train  17.755% | test  18.451% | mass  32.98% | drift  33.96% | data 2.15e-03 | w_eik 4.478e-01 | 2.8m
  1500 loss 3.963e-03 | train  14.513% | test  15.263% | mass  24.65% | drift  25.36% | data 9.75e-04 | w_eik 3.959e-01 | 3.3m
  1750 loss 2.997e-03 | train  12.188%

In [7]:
!python train_operator.py --loss strong --saw --saw_init zero --n_labelled 8 --steps 20000 --n_train 16 --n_test 100 --seed 43

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 8/16 instances  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 1.638e-02 | train  33.851% | test  34.178% | mass  63.06% | drift  65.14% | data 1.14e-02 | w_eik 1.581e-01 | 0.6m
   500 loss 2.032e-02 | train  33.944% | test  34.383% | mass  50.51% | drift  52.20% | data 1.48e-02 | w_eik 3.439e-01 | 1.1m
   750 loss 1.768e-02 | train  33.328% | test  33.917% | mass  43.48% | drift  44.94% | data 1.31e-02 | w_eik 4.878e-01 | 1.7m
  1000 loss 1.035e-02 | train  27.348% | test  28.243% | mass  44.67% | drift  46.19% | data 5.42e-03 | w_eik 5.429e-01 | 2.2m
  1250 loss 7.900e-03 | train  23.669% | test  24.836% | mass  40.12% | drift  41.51% | data 2.66e-03 | w_eik 5.112e-01 | 2.8m
  1500 loss 5.224e-03 | train  17.962% | test  19.418% | mass  36.79% | drift  38.09% | data 1.66e-03 | w_eik 4.389e-01 | 3.4m
  1750 loss 2.423e-03 | train  12.581%

In [8]:
!python train_operator.py --loss strong --saw --saw_init zero --n_labelled 8 --steps 20000 --n_train 16 --n_test 100 --seed 44

device=cuda  grid=64x64x32  loss=strong
benchmark: RO  (velocity fixed)
SAW: q=0.95, beta=0.999, init=zero
labels: 8/16 instances  lambda_data=1
IC: hard (structural)
params=7.38M  train=16  test=100
   250 loss 1.642e-02 | train  33.977% | test  34.042% | mass  46.00% | drift  47.46% | data 1.06e-02 | w_eik 1.441e-01 | 0.6m
   500 loss 1.630e-02 | train  33.466% | test  33.560% | mass  46.90% | drift  48.39% | data 1.12e-02 | w_eik 3.326e-01 | 1.1m
   750 loss 1.988e-02 | train  33.368% | test  33.499% | mass  47.84% | drift  49.36% | data 1.52e-02 | w_eik 4.799e-01 | 1.7m
  1000 loss 1.984e-02 | train  27.995% | test  28.227% | mass  45.39% | drift  46.83% | data 1.40e-02 | w_eik 5.613e-01 | 2.2m
  1250 loss 1.439e-02 | train  21.778% | test  22.088% | mass  37.10% | drift  38.27% | data 8.72e-03 | w_eik 5.257e-01 | 2.8m
  1500 loss 8.813e-03 | train  16.887% | test  17.315% | mass  32.25% | drift  33.27% | data 4.80e-03 | w_eik 4.456e-01 | 3.3m
  1750 loss 2.695e-03 | train  11.078%

In [9]:
!python collect_results.py

arm         v        n       grid  steps sd    train     test     mass    min
-----------------------------------------------------------------------------
hybrid-8    fixed   16    64^2x32  20000 42   1.184%   1.845%    4.10%   44.5
hybrid-8    fixed   16    64^2x32  20000 43   1.077%   1.875%    4.04%   44.8
hybrid-8    fixed   16    64^2x32  20000 44   0.938%   1.656%    3.38%   44.6
data-free   fixed   16    64^2x32  20000 42   4.930%   5.258%    5.50%   45.0
data-free   fixed   16    64^2x32  20000 43   2.956%   3.258%    4.78%   45.0
data-free   fixed   16    64^2x32  20000 44   5.254%   5.632%    4.03%   43.7
supervised  fixed   16    64^2x32  20000 42   0.402%   2.441%    7.97%   41.7
supervised  fixed   16    64^2x32  20000 43   0.429%   2.611%    8.87%   41.7
supervised  fixed   16    64^2x32  20000 44   0.361%   2.811%   10.54%   41.8

arm         v        n  steps seeds  test mean     std  mass mean     std
-------------------------------------------------------------------

---
## Results

In [10]:
!python collect_results.py --sort test

arm         v        n       grid  steps sd    train     test     mass    min
-----------------------------------------------------------------------------
hybrid-8    fixed   16    64^2x32  20000 44   0.938%   1.656%    3.38%   44.6
hybrid-8    fixed   16    64^2x32  20000 42   1.184%   1.845%    4.10%   44.5
hybrid-8    fixed   16    64^2x32  20000 43   1.077%   1.875%    4.04%   44.8
supervised  fixed   16    64^2x32  20000 42   0.402%   2.441%    7.97%   41.7
supervised  fixed   16    64^2x32  20000 43   0.429%   2.611%    8.87%   41.7
supervised  fixed   16    64^2x32  20000 44   0.361%   2.811%   10.54%   41.8
data-free   fixed   16    64^2x32  20000 43   2.956%   3.258%    4.78%   45.0
data-free   fixed   16    64^2x32  20000 42   4.930%   5.258%    5.50%   45.0
data-free   fixed   16    64^2x32  20000 44   5.254%   5.632%    4.03%   43.7

arm         v        n  steps seeds  test mean     std  mass mean     std
-------------------------------------------------------------------

In [11]:
!python eval_checkpoint.py strong_saw_saw0_n16_20k_64x32_s42 --n_test 100


strong_saw_saw0_n16_20k_64x32_s42   loss=strong  IC=hard  n_train=16  grid=64^2 x 32  bench=RO  v=fixed  n_test=100
set         rel L2     abs L2   mass(ref)  mass(pi R^2)     drift
-----------------------------------------------------------------
train       4.930%  1.716e-02       4.74%         4.76%     5.28%
test        5.258%  1.827e-02       5.50%         5.50%     5.63%
(exact)     0.000%  0.000e+00       0.00%         0.67%     1.02%


In [12]:
!python eval_checkpoint.py supervised_n16_20k_64x32_s42 --n_test 100


supervised_n16_20k_64x32_s42   loss=supervised  IC=hard  n_train=16  grid=64^2 x 32  bench=RO  v=fixed  n_test=100
set         rel L2     abs L2   mass(ref)  mass(pi R^2)     drift
-----------------------------------------------------------------
train       0.402%  1.387e-03       0.77%         0.91%     1.15%
test        2.441%  8.572e-03       7.97%         7.93%     8.25%
(exact)     0.000%  0.000e+00       0.00%         0.67%     1.02%


In [13]:
!python eval_checkpoint.py strong_saw_lab8_saw0_n16_20k_64x32_s42 --n_test 100


strong_saw_lab8_saw0_n16_20k_64x32_s42   loss=strong  IC=hard  n_train=16  grid=64^2 x 32  bench=RO  v=fixed  n_test=100
set         rel L2     abs L2   mass(ref)  mass(pi R^2)     drift
-----------------------------------------------------------------
train       1.184%  4.164e-03       2.63%         2.70%     2.87%
test        1.845%  6.430e-03       4.10%         4.06%     4.23%
(exact)     0.000%  0.000e+00       0.00%         0.67%     1.02%


### Figures

Contours are what matter: a relative L2 number cannot distinguish a slightly
displaced interface from a smeared one.

In [14]:
!python visualize.py strong_saw_saw0_n16_20k_64x32_s42 supervised_n16_20k_64x32_s42 strong_saw_lab8_saw0_n16_20k_64x32_s42 --n_test 100

plotting instance 9: p=[0.203 1.608 0.124 1.    0.5   0.5  ] omega=1.000
  physics only           mean 5.258%  this instance 4.886%
  supervised             mean 2.441%  this instance 0.428%
  hybrid (8 labels)      mean 1.845%  this instance 0.438%

wrote interfaces_strong_saw_saw0_n16_20k_64x32_s42.png, fields_strong_saw_saw0_n16_20k_64x32_s42.png, spread_strong_saw_saw0_n16_20k_64x32_s42.png
